# web scraper
trying out a web scraper to pull required details from the open library

In [2]:
# load required libraries
import requests
from re import search, sub

In [3]:
# retrieve book details from URL
# start by getting book ID from URL
url_in = 'https://openlibrary.org/books/OL28495417M/Hogfather' # hogfather by terry pratchett
match = search(r'(OL\d+[A-Z])', url_in)
book_id = match.group(1) if match else None

# include application header in API request
headers = {
    "User-Agent": "EEcordBookClubBot/0.1 (lorelei@duck.com)"
}
response = requests.get(f'https://openlibrary.org/books/{book_id}.json', headers=headers)
book_data = response.json()

author_names = []
author_references = book_data.get('authors', [])
    
for ref in author_references:
    author_key = ref.get('key') or ref.get('author', {}).get('key')
    if author_key:
        # Call the Author API
        author_res = requests.get(f"https://openlibrary.org{author_key}.json")
        if author_res.status_code == 200:
            author_names.append(author_res.json().get('name'))

print(f'Retrieved details for {book_data.get('title')} by {author_names[0]} ({book_data.get('publish_date')}, {book_data.get('number_of_pages')} pages).')

Retrieved details for Hogfather by Terry Pratchett (2013, 432 pages).


In [4]:
# repeat but with a plaintext search
search_term = 'rf kuang'
search_str = sub(r'\s', '+', search_term)

print(f'https://openlibrary.org/search.json?q={search_str}')
response = requests.get(f'https://openlibrary.org/search.json?q={search_str}', headers=headers)
search_results = response.json()

if search_results.get('docs'):
    book_data = search_results.get('docs')[0]
    pages = "N/A"
    
    # 1. Get the Work key (e.g., /works/OL123W)
    work_key = book_data.get('key') 
    
    if work_key:
        # 2. Query the /editions endpoint for this work
        editions_url = f"https://openlibrary.org{work_key}/editions.json"
        ed_res = requests.get(editions_url, headers=headers)
        
        if ed_res.status_code == 200:
            editions_data = ed_res.json()
            # 3. Look through the editions for the first one with a page count
            for entry in editions_data.get('entries', []):
                if 'number_of_pages' in entry:
                    pages = entry['number_of_pages']
                    break # Stop once we find a valid page count

    title = book_data.get("title")
    author = book_data.get("author_name", ["Unknown"])[0]
    year = book_data.get("first_publish_year", "N/A")

    print(f'Retrieved details for {title} by {author} ({year}, {pages} pages).')

https://openlibrary.org/search.json?q=rf+kuang
Retrieved details for Yellowface by R. F. Kuang (2023, 352 pages).


In [ ]:
# rewrite as function
def get_book_details(search_input):
    # define header to include in API call
    headers = {
    "User-Agent": "EEcordBookClubBot/0.1 (lorelei@duck.com)"
    }
    
    # check whether search_input is a URL or a plaintext search string
    input_is_url = 'openlibrary.org/' in search_input

    if input_is_url:
        # extract book ID from URL
        match = search(r'(OL\d+[A-Z])', url_in)
        
        if not match:
            raise ValueError("Supplied URL didn't include a valid Open Library book ID.\nPlease supply a valid Open Library book URL, or provide a plaintext search term.")
        
        # call API for matching ID
        book_id = match.group(1)
        book_data = requests.get(f'https://openlibrary.org/books/{book_id}.json', headers=headers).json()

        if not search_results.get('title'):
            raise ValueError("Supplied URL didn't match a valid Open Library URL.\nPlease supply a valid Open Library book URL, or supply a plaintext search term.")

        # pull first author name for book
        author_names = []
        author_references = book_data.get('authors', [])
    
        for ref in author_references:
            author_key = ref.get('key') or ref.get('author', {}).get('key')
            if author_key:
                # Call the Author API
                author_res = requests.get(f"https://openlibrary.org{author_key}.json")
                if author_res.status_code == 200:
                    author_names.append(author_res.json().get('name'))

        title = book_data.get('title')
        authors = ','.join(author_names)
        publish_year = book_data.get('publish_date')
        pages = book_data.get('number_of_pages')

        book_details = {
            'title': title,
            'authors': authors,
            'publish_year': publish_year,
            'pages': pages
            }

        return(book_details)
    
    else:
        search_str = sub(r'\s', '+', search_input)
        
        search_results = requests.get(f'https://openlibrary.org/search.json?q={search_str}', headers=headers).json()
        
        if not search_results.get('docs'):
            raise ValueError("No results for search term.\nPlease try another search term, or supply a valid Open Library URL.")

    if search_results.get('docs'):
        book_data = search_results.get('docs')[0]
        pages = "N/A"
    
        work_key = book_data.get('key') 
    
        if work_key:
            editions_url = f"https://openlibrary.org{work_key}/editions.json"
        editions_data = requests.get(editions_url, headers=headers).json()
        
        for entry in editions_data.get('entries', []):
            if 'number_of_pages' in entry:
                pages = entry['number_of_pages']
                break

        title = book_data.get("title")
        authors = ','.join(book_data.get("author_name", ["Unknown"]))
        publish_year = book_data.get("first_publish_year", "N/A")

        book_details = {
            'title': title,
            'authors': authors,
            'publish_year': publish_year,
            'pages': pages
            }
        
        return(book_details)
    
print(get_book_details('schopenhauer cure'))
# need to add in a URL field in the output, maybe a blurb?

{'title': 'The Schopenhauer cure', 'authors': 'Irvin D. Yalom', 'publish_year': 2005, 'pages': 358}
